In [13]:
# Zachary Katz
# zachary_katz@mines.edu
# 07 May 2025
# Gigi Albers
# Imports

%load_ext autoreload
%autoreload 2

import util.plotting_helpers as plothelp
import matplotlib.pyplot as plt
import earthaccess
import xarray as xr
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pyproj import Transformer
from scipy import interpolate
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut



auth = earthaccess.login(strategy="netrc")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [18]:
geolocator = Nominatim(user_agent="swot_granule_app")

def get_bounding_box_from_placename(place_name):
    """
    Geocodes a place name to find its bounding box coordinates.
    Returns a tuple (min_lon, min_lat, max_lon, max_lat) or None.
    """
    try:
        # Use geocode to get location details.
        # The `viewbox` parameter can help narrow down search results if you have a general area.
        # The `timeout` is useful for preventing the code from hanging.
        location = geolocator.geocode(place_name, exactly_one=True, timeout=10)
        
        if location:
            # The boundingbox is returned as a list of strings: [min_lat, max_lat, min_lon, max_lon]
            bbox = [float(coord) for coord in location.raw['boundingbox']]
            # We need to reorder it to the format earthaccess expects: (min_lon, min_lat, max_lon, max_lat)
            min_lat, max_lat, min_lon, max_lon = bbox
            print(f"Found bounding box for {place_name}: {min_lon}, {min_lat}, {max_lon}, {max_lat}")
            return (min_lon, min_lat, max_lon, max_lat)
        else:
            print(f"Could not find a location for '{place_name}'. Please try a different name.")
            return None
    except GeocoderTimedOut:
        print("Geocoding service timed out. Please try again.")
        return None
    except Exception as e:
        print(f"An error occurred during geocoding: {e}")
        return None

# Your existing search and open function, now simplified
def search_and_open_swot_data(bounding_box, start_date, end_date):
    """
    Searches for and opens SWOT data for a given bounding box and temporal range.
    Returns an xarray Dataset or None if no granules are found.
    """
    results = earthaccess.search_data(
        short_name="SWOT_L2_HR_Raster_D",
        bounding_box=bounding_box,
        temporal=(start_date, end_date),
    )

    if not results:
        print(f"  No granules found for the specified dates and area.")
        return None
    
    if len(results) > 1:
        print(f"  Warning: Found {len(results)} granules. Opening the first one.")
        # Consider adding more robust logic for handling multiple granules here
        return xr.open_dataset(earthaccess.open(results)[0], engine="h5netcdf")
    else:
        print(f"  Found 1 granule. Opening...")
        return xr.open_dataset(earthaccess.open(results)[0], engine="h5netcdf")


In [19]:
def get_overlapping_dates(all_granules, before_date_str):
    # Exclude the before_date from the list of all dates
    after_dates = sorted(list(set(g.temporal[0][:10] for g in all_granules if g.temporal[0][:10] != before_date_str)))
    return after_dates

In [20]:
if __name__ == "__main__":
    num_pairs = int(input("How many SWOT granule pairs do you want to analyze? "))

    if num_pairs <= 0:
        print("No pairs requested. Exiting.")
        exit()

    place_name = input("\n--- Enter the name of the place to analyze (e.g., 'Lower Mississippi River, USA'): ")
    bounding_box = get_bounding_box_from_placename(place_name)

    if not bounding_box:
        print("Cannot proceed without a valid bounding box. Exiting.")
        exit()

    # Step 1: Search for ALL granules in the AOI to get a master list
    print("\nSearching for all SWOT granules in the specified area...")
    all_granules = earthaccess.search_data(
        short_name="SWOT_L2_HR_Raster_D",
        bounding_box=bounding_box,
        temporal=('2023-01-01', '2025-12-31') # Use a wide temporal range for a complete list
    )

    if not all_granules:
        print("No granules found in the specified area over the entire SWOT mission. Exiting.")
        exit()

    # Step 2: Extract unique dates from the master list for the user to choose from
    # CORRECTED LINE: Using g.data_start instead of g.temporal
    unique_dates = sorted(list(set(g.data_start[:10] for g in all_granules)))
    print("\n--- Found the following dates with SWOT granules in your area: ---")
    for i, date in enumerate(unique_dates):
        print(f"  {i+1}: {date}")

    swot_datasets = []

    # Step 3: Loop for each pair the user wants to analyze
    for i in range(num_pairs):
        print(f"\n--- Collecting data for Pair {i+1} ---")

        # User selects the "before" date
        while True:
            try:
                choice = int(input(f"Please select the 'BEFORE' date by number (1-{len(unique_dates)}): "))
                if 1 <= choice <= len(unique_dates):
                    before_date_str = unique_dates[choice - 1]
                    break
                else:
                    print("Invalid choice. Please enter a number from the list.")
            except ValueError:
                print("Invalid input. Please enter a number.")

        # Filter the master list to get the granule(s) for the chosen before_date
        # Use g.data_start here as well
        before_granules = [g for g in all_granules if g.data_start[:10] == before_date_str]

        if not before_granules:
            print("No granule found for the selected 'before' date. Skipping pair.")
            continue
        ds_before = xr.open_dataset(earthaccess.open(before_granules)[0], engine="h5netcdf")
        print(f"Downloaded 'before' granule for {before_date_str}.")

        # Step 4: Find and present the list of overlapping 'after' dates
        after_dates = get_overlapping_dates(all_granules, before_date_str)
        if not after_dates:
            print("No other overlapping dates found. Skipping this pair.")
            continue

        print("\n--- Select an 'AFTER' date from the following overlapping passes: ---")
        for j, date in enumerate(after_dates):
            print(f"  {j+1}: {date}")

        # User selects the "after" date
        while True:
            try:
                choice = int(input(f"Please select the 'AFTER' date by number (1-{len(after_dates)}): "))
                if 1 <= choice <= len(after_dates):
                    after_date_str = after_dates[choice - 1]
                    break
                else:
                    print("Invalid choice. Please enter a number from the list.")
            except ValueError:
                print("Invalid input. Please enter a number.")

        # Step 5: Get the "after" granule based on the user's choice
        # Use g.data_start here as well
        after_granules = [g for g in all_granules if g.data_start[:10] == after_date_str]
        if not after_granules:
            print("No granule found for the selected 'after' date. Skipping pair.")
            continue
        ds_after = xr.open_dataset(earthaccess.open(after_granules)[0], engine="h5netcdf")
        print(f"Downloaded 'after' granule for {after_date_str}.")

        swot_datasets.append((ds_before, ds_after))

How many SWOT granule pairs do you want to analyze?  1

--- Enter the name of the place to analyze (e.g., 'Lower Mississippi River, USA'):  Florida


Found bounding box for Florida: -87.634896, 24.396308, -79.974306, 31.000762

Searching for all SWOT granules in the specified area...


AttributeError: 'DataGranule' object has no attribute 'data_start'

In [17]:
def plot_swot_data(ax, ds, title, vmin, vmax):
    if ds is None:
        ax.set_title(f"{title}\n(No data available)", fontsize=14)
        ax.set_visible(False) # Hide axis if no data
    return None, None

    # Get data
wse = ds["wse"] + ds["height_cor_xover"]
    
    # Convert UTM to lat/lon
utm_zone = ds.utm_zone_num
utm_crs = ccrs.UTM(zone=utm_zone, southern_hemisphere=False)
transformer = Transformer.from_crs(utm_crs, ccrs.PlateCarree(), always_xy=True)
x_utm, y_utm = wse["x"].values, wse["y"].values
X_utm, Y_utm = np.meshgrid(x_utm, y_utm)
X_lon, Y_lat = transformer.transform(X_utm, Y_utm)
    
    # Plot data
mesh = ax.pcolormesh(
    X_lon,
    Y_lat,
    wse.values,
    transform=ccrs.PlateCarree(),
    cmap="viridis",
    vmin=vmin,
    vmax=vmax,
)
    
    # Add map features
ax.gridlines(draw_labels=True)
ax.add_feature(cfeature.LAND, facecolor='lightgray')
ax.add_feature(cfeature.OCEAN, facecolor='lightblue')
ax.coastlines(resolution='10m', linewidth=1)
ax.set_title(title, fontsize=14)
    
return mesh, wse


NameError: name 'ds' is not defined

In [8]:
def plot_difference(ax, ds_before, ds_after, title, vmin, vmax):
    """
    Calculates and plots the difference between two SWOT WSE datasets.
    Returns the pcolormesh object.
    """
    if ds_before is None or ds_after is None:
        ax.set_title(f"{title}\n(Cannot compute difference due to missing data)", fontsize=14)
        ax.set_visible(False) # Hide axis if no data
        return None

    wse_before = ds_before["wse"] + ds_before["height_cor_xover"]
    wse_after = ds_after["wse"] + ds_after["height_cor_xover"]

    # Handle differing UTM zones - reproject 'before' onto 'after's grid for difference calculation
    # For a robust solution, you might want to reproject both to a common CRS if they are very different.
    if ds_before.utm_zone_num != ds_after.utm_zone_num:
        print(f"Warning: UTM zones differ (Before: {ds_before.utm_zone_num}, After: {ds_after.utm_zone_num}). Reprojecting 'before' to 'after's CRS for difference calculation.")
        # Create transformers for both datasets
        transformer_before_to_latlon = Transformer.from_crs(
            ccrs.UTM(zone=ds_before.utm_zone_num, southern_hemisphere=False),
            ccrs.PlateCarree(),
            always_xy=True
        )
        transformer_after_to_latlon = Transformer.from_crs(
            ccrs.UTM(zone=ds_after.utm_zone_num, southern_hemisphere=False),
            ccrs.PlateCarree(),
            always_xy=True
        )

        # Get lat/lon for both datasets
        x_before_utm, y_before_utm = wse_before["x"].values, wse_before["y"].values
        X_before_utm, Y_before_utm = np.meshgrid(x_before_utm, y_before_utm)
        lon_before, lat_before = transformer_before_to_latlon.transform(X_before_utm, Y_before_utm)

        x_after_utm, y_after_utm = wse_after["x"].values, wse_after["y"].values
        X_after_utm, Y_after_utm = np.meshgrid(x_after_utm, y_after_utm)
        lon_after, lat_after = transformer_after_to_latlon.transform(X_after_utm, Y_after_utm)

        # Create xarray DataArrays with lat/lon coordinates for interpolation
        wse_before_ll = xr.DataArray(
            wse_before.values,
            coords={'lat': lat_before[:,0], 'lon': lon_before[0,:]}, # Assuming rectilinear grid
            dims=['y', 'x']
        ).rename({'y': 'lat', 'x': 'lon'})

        wse_after_ll = xr.DataArray(
            wse_after.values,
            coords={'lat': lat_after[:,0], 'lon': lon_after[0,:]},
            dims=['y', 'x']
        ).rename({'y': 'lat', 'x': 'lon'})

        # Interpolate 'before' onto 'after's lat/lon grid
        try:
            wse_before_interpolated = wse_before_ll.interp(lat=wse_after_ll.lat, lon=wse_after_ll.lon, method="linear")
            wse_diff = wse_after_ll - wse_before_interpolated
        except Exception as e:
            print(f"Error during interpolation for difference: {e}. Cannot compute difference with differing UTM zones.")
            ax.set_title(f"{title}\n(Error in re-gridding)", fontsize=14)
            ax.set_visible(False)
            return None
        
        # Use the 'after' grid for plotting the difference
        X_lon, Y_lat = lon_after, lat_after
        utm_crs_plot = ccrs.UTM(zone=ds_after.utm_zone_num, southern_hemisphere=False) # Use after's zone for plotting CRS
    else:
        # If UTM zones are the same, proceed with direct difference on original grid if aligned,
        # or interpolate if necessary (e.g., if coordinates are slightly off)
        try:
            # Attempt to interpolate before onto after's grid for precise alignment
            wse_before_aligned = wse_before.interp(x=wse_after.x, y=wse_after.y, method="linear")
            wse_diff = wse_after - wse_before_aligned
        except Exception as e:
            print(f"Error interpolating WSE 'before' onto 'after' grid: {e}. Proceeding with direct subtraction (assuming perfect grid alignment).")
            # Fallback to direct subtraction if interpolation fails, assuming similar grids
            if wse_before.shape == wse_after.shape and np.allclose(wse_before['x'], wse_after['x']) and np.allclose(wse_before['y'], wse_after['y']):
                wse_diff = wse_after - wse_before
            else:
                print("Direct difference not possible due to incompatible grids. Skipping difference plot.")
                ax.set_title(f"{title}\n(Incompatible data grids)", fontsize=14)
                ax.set_visible(False)
                return None
        
        utm_crs_plot = ccrs.UTM(zone=ds_before.utm_zone_num, southern_hemisphere=False) # Use either zone
        transformer = Transformer.from_crs(utm_crs_plot, ccrs.PlateCarree(), always_xy=True)
        x_utm, y_utm = wse_after["x"].values, wse_after["y"].values # Use after's grid for plotting
        X_utm, Y_utm = np.meshgrid(x_utm, y_utm)
        X_lon, Y_lat = transformer.transform(X_utm, Y_utm)

    mesh = ax.pcolormesh(
        X_lon,
        Y_lat,
        wse_diff.values,
        transform=ccrs.PlateCarree(),
        cmap="RdBu", # Diverging colormap for differences
        vmin=vmin,
        vmax=vmax,
    )

    ax.gridlines(draw_labels=True)
    ax.add_feature(cfeature.LAND, facecolor='lightgray')
    ax.add_feature(cfeature.OCEAN, facecolor='lightblue')
    ax.coastlines(resolution='10m', linewidth=1)
    ax.set_title(title, fontsize=14)

    return mesh

In [9]:
num_pairs = int(input("How many SWOT granule pairs (before and after) do you want to analyze? "))

swot_datasets = [] # To store tuples of (ds_before, ds_after)

for i in range(num_pairs):
    print(f"\n--- Collecting data for Pair {i+1} ---")
    ds_before = get_swot_granule_data("BEFORE")
    ds_after = get_swot_granule_data("AFTER")
    
    if ds_before is not None and ds_after is not None:
        swot_datasets.append((ds_before, ds_after))
    elif ds_before is None and ds_after is None:
        print(f"Skipping Pair {i+1} as no granules were found for both before and after.")
    elif ds_before is None:
        print(f"Skipping Pair {i+1} as no 'before' granule was found.")
    else: # ds_after is None
        print(f"Skipping Pair {i+1} as no 'after' granule was found.")

How many SWOT granule pairs (before and after) do you want to analyze?  1



--- Collecting data for Pair 1 ---


NameError: name 'get_swot_granule_data' is not defined

In [ ]:
if not swot_datasets:
    print("No valid granule pairs were found or entered. Exiting.")
else:
    # Determine overall figure size and grid based on number of pairs
    # Each pair will have 3 plots (before, after, difference) + 2 colorbars (one for WSE, one for Difference)
    # We'll put colorbars to the right of their respective plots.
    num_rows = num_pairs
    num_cols = 3 # Before, After, Difference
    
    # Each row will have 3 main plots, plus two narrow columns for colorbars
    fig = plt.figure(figsize=(24, 8 * num_rows)) # Adjust figsize dynamically for better readability
    
    # Define grid spec. For each row: plot1, plot2, plot3, wse_colorbar_space, diff_colorbar_space
    gs = fig.add_gridspec(num_rows, num_cols + 2, width_ratios=[1, 1, 1, 0.05, 0.05], wspace=0.3)
    
    main_location = input("Enter a main location title for all plots (e.g., 'Lower Mississippi River'): ")

    for i, (ds_before, ds_after) in enumerate(swot_datasets):
        print(f"\n--- Plotting for Pair {i+1} ---")
        # Determine plot titles based on user input for each pair
        title_before = input(f"Title for Before Event (Pair {i+1}) (e.g., Location, mm/dd/yyyy): ")
        title_after = input(f"Title for After Event (Pair {i+1}) (e.g., Location, mm/dd/yyyy): ")
        title_diff = input(f"Title for Difference Plot (Pair {i+1}) (e.g., Change in WSE, mm/dd/yyyy): ")

        # Before Plot
        ax1 = fig.add_subplot(gs[i, 0], projection=ccrs.PlateCarree())
        mesh1, wse1 = plot_swot_data(ax1, ds_before, title_before, vmin=0, vmax=10) # Common WSE range

        # After Plot
        ax2 = fig.add_subplot(gs[i, 1], projection=ccrs.PlateCarree())
        mesh2, wse2 = plot_swot_data(ax2, ds_after, title_after, vmin=0, vmax=10) # Common WSE range

        # Difference Plot
        ax3 = fig.add_subplot(gs[i, 2], projection=ccrs.PlateCarree())
        mesh3 = plot_difference(ax3, ds_before, ds_after, title_diff, vmin=-5, vmax=5) # Symmetric range for difference

        # WSE Colorbar (for Before/After plots in this row)
        if mesh1: # Only add if the plot was successful
            cax_wse = fig.add_subplot(gs[i, num_cols]) # Position for WSE colorbar
            cbar_wse = fig.colorbar(mesh1, cax=cax_wse, orientation='vertical')
            cbar_wse.set_label("Water Surface Elevation [m]", fontsize=12, labelpad=15, rotation=90)
        
        # Difference Colorbar (for the Difference plot in this row)
        if mesh3: # Only add if the plot was successful
            cax_diff = fig.add_subplot(gs[i, num_cols + 1]) # Position for Difference colorbar
            cbar_diff = fig.colorbar(mesh3, cax=cax_diff, orientation='vertical')
            cbar_diff.set_label("WSE Difference [m]", fontsize=12, labelpad=15, rotation=90)

    # Add the main title for the entire figure
    fig.text(
        x=0.5,
        y=0.98, # Adjust y position for the main title
        s=main_location,
        fontsize=20,
        color='black',
        va='top',
        ha='center'
    )

    #plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to make space for the main title
    plt.show()